# ConvoGuard: Phase 2 — Published BERT Baselines (BERT-SC & BERT-SC+)

**Task:** Early Forecasting of Personal Attacks in Wikipedia Talk-Page Discussions  
**Dataset:** Conversations Gone Awry (CGA-Wiki)  
**Reference Paper:** *Dynamic Forecasting of Conversation Derailment*, Yova Kementchedjhieva & Anders Søgaard (EMNLP 2021)  
**Official Reference:** https://github.com/YovaKem/forecasting_derailment

---
### Protocol & Non-Negotiable Rules
1. **No Test Set Access:** Only `train_conversations.parquet` and `val_conversations.parquet` are loaded. The official test split is **never** loaded or evaluated.
2. **Pre-Target Forecasting:** All conversational inputs **strictly exclude** the terminal target message.
3. **Turn-Preserving Truncation:** Subwords beyond 512 tokens are cropped by recursively removing tokens from the longest turn, preserving all dialogue turns.
4. **Validation Checkpointing:** All model selections and decision thresholds are selected solely on validation data.
5. **Reproducible Baselines:**
   - **BERT-SC (Static):** One pre-target sequence per conversation.
   - **BERT-SC+ (Dynamic):** Successive prefix sequences per conversation with forecast-horizon evaluation.

In [ ]:
# Cell 1: Environment & GPU Verification
import random
import sys

import numpy as np
import torch


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

seed_everything(42)

print('=== System Environment ===')
print(f'Python version: {sys.version.split()[0]}')
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')
    print(f'GPU Count: {torch.cuda.device_count()}')
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'Total VRAM: {vram_gb:.2f} GB')
else:
    print('WARNING: Running on CPU. For training BERT-SC models, enable GPU Accelerator in Kaggle settings.')

In [ ]:
# Cell 2: Kaggle Dataset Discovery
from pathlib import Path

# Search common Kaggle input directories or local fallback
candidate_dirs = [
    Path('/kaggle/input/convoguard-data/raw'),
    Path('/kaggle/input/cga-wiki/raw'),
    Path('/kaggle/input/convoguard-cga-wiki'),
    Path('/kaggle/input/conversations-gone-awry'),
    Path('data/raw'),
    Path('../data/raw'),
]

data_root = None
for d in candidate_dirs:
    if d.exists() and (d / 'train_conversations.parquet').exists():
        data_root = d
        break

if data_root is None:
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        for p in kaggle_input.rglob('train_conversations.parquet'):
            data_root = p.parent
            break

if data_root is None:
    raise FileNotFoundError(
        'Could not locate train_conversations.parquet. Please attach the CGA-Wiki dataset to this Kaggle notebook.'
    )

train_path = data_root / 'train_conversations.parquet'
val_path = data_root / 'val_conversations.parquet'
test_path = data_root / 'test_conversations.parquet'

print(f'[*] Data root located: {data_root.resolve()}')
print(f'[*] Train split found: {train_path.exists()} ({train_path.stat().st_size:,} bytes)')
print(f'[*] Val split found:   {val_path.exists()} ({val_path.stat().st_size:,} bytes)')
print(f'[*] Test split exists on disk: {test_path.exists()} -> EXCLUDED from loading per protocol!')

In [ ]:
# Cell 3: Install and Import ConvoGuard
import subprocess

# If running on Kaggle, clone or import convoguard
repo_dir = Path('ConvoGuard')
if not (Path('src/convoguard').exists() or (repo_dir / 'src/convoguard').exists()):
    print('[*] Cloning ConvoGuard repository from GitHub...')
    subprocess.run(['git', 'clone', 'https://github.com/nayanojwal0810/ConvoGuard.git'], check=True)
    sys.path.insert(0, str((repo_dir / 'src').resolve()))
else:
    if Path('src/convoguard').exists():
        sys.path.insert(0, str(Path('src').resolve()))
    elif (repo_dir / 'src/convoguard').exists():
        sys.path.insert(0, str((repo_dir / 'src').resolve()))

try:
    import transformers
except ImportError:
    print('[*] Installing transformers...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'transformers'], check=True)

import pandas as pd
from transformers import BertTokenizer

from convoguard.data.forecasting import build_dynamic_examples, build_static_examples
from convoguard.data.loader import load_split
from convoguard.models.bert import (
    BertForecastingClassifier,
    BertForecastingDataset,
    pad_bert_collate,
)
from convoguard.models.evaluation import (
    aggregate_dynamic_predictions,
    compute_forecast_horizon_metrics,
    evaluate_predictions,
)

print('[OK] ConvoGuard forecasting modules successfully imported!')

In [ ]:
# Cell 4: Load Data and Build Static & Dynamic Forecasting Examples
print('[*] Loading raw Train and Validation splits...')
df_train = load_split(train_path, validate=True)
df_val = load_split(val_path, validate=True)

print(f'    Train rows: {len(df_train):,}, unique convs: {df_train["conversation_id"].nunique():,}')
print(f'    Val rows:   {len(df_val):,}, unique convs: {df_val["conversation_id"].nunique():,}')

print('[*] Building static examples (1 pre-target sequence per conversation)...')
static_train = build_static_examples(df_train)
static_val = build_static_examples(df_val)
print(f'    Static train examples: {len(static_train):,} (derailment rate: {static_train["label"].mean():.2%})')
print(f'    Static val examples:   {len(static_val):,} (derailment rate: {static_val["label"].mean():.2%})')

print('[*] Building dynamic examples (successive prefix sequences)...')
dynamic_train = build_dynamic_examples(df_train)
dynamic_val = build_dynamic_examples(df_val)
print(f'    Dynamic train examples: {len(dynamic_train):,} prefixes')
print(f'    Dynamic val examples:   {len(dynamic_val):,} prefixes')

# Contract Verification: Verify terminal target comments are never included
assert not any('TARGET ATTACK' in text for text in static_train['text']), 'Leakage detected!'
print('[OK] Pre-target integrity verified: 0 target turns present in input features.')

In [ ]:
# Cell 5: Hyperparameters and Smoke-Test Configuration
# Set SMOKE_TEST = True to test execution in under 2 minutes
SMOKE_TEST = False

CONFIG = {
    "model_name": "bert-base-uncased",
    "max_length": 512,
    "batch_size": 16,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "epochs": 1 if SMOKE_TEST else 3,
    "seed": 42,
    "output_dir": Path("outputs_bert_baselines"),
}

CONFIG["output_dir"].mkdir(parents=True, exist_ok=True)
(CONFIG["output_dir"] / "checkpoints").mkdir(exist_ok=True)

if SMOKE_TEST:
    print("[!] SMOKE_TEST enabled: sub-sampling 128 train and 64 val examples for rapid syntax check.")
    train_static_data = static_train.sample(n=min(128, len(static_train)), random_state=42).to_dict('records')
    val_static_data = static_val.sample(n=min(64, len(static_val)), random_state=42).to_dict('records')
    train_dynamic_data = dynamic_train.sample(n=min(256, len(dynamic_train)), random_state=42).to_dict('records')
    val_dynamic_data = dynamic_val.sample(n=min(128, len(dynamic_val)), random_state=42).to_dict('records')
else:
    train_static_data = static_train.to_dict('records')
    val_static_data = static_val.to_dict('records')
    train_dynamic_data = dynamic_train.to_dict('records')
    val_dynamic_data = dynamic_val.to_dict('records')

print(f"[*] Initializing {CONFIG['model_name']} tokenizer...")
tokenizer = BertTokenizer.from_pretrained(CONFIG["model_name"])
print("[OK] Tokenizer ready.")

In [ ]:
# Cell 6: PyTorch DataLoaders
from torch.utils.data import DataLoader

ds_static_train = BertForecastingDataset(train_static_data, tokenizer, max_length=CONFIG["max_length"])
ds_static_val = BertForecastingDataset(val_static_data, tokenizer, max_length=CONFIG["max_length"])

ds_dynamic_train = BertForecastingDataset(train_dynamic_data, tokenizer, max_length=CONFIG["max_length"])
ds_dynamic_val = BertForecastingDataset(val_dynamic_data, tokenizer, max_length=CONFIG["max_length"])

collate_fn = lambda b: pad_bert_collate(b, pad_token_id=tokenizer.pad_token_id or 0)

loader_static_train = DataLoader(ds_static_train, batch_size=CONFIG["batch_size"], shuffle=True, collate_fn=collate_fn)
loader_static_val = DataLoader(ds_static_val, batch_size=CONFIG["batch_size"], shuffle=False, collate_fn=collate_fn)

loader_dynamic_train = DataLoader(ds_dynamic_train, batch_size=CONFIG["batch_size"], shuffle=True, collate_fn=collate_fn)
loader_dynamic_val = DataLoader(ds_dynamic_val, batch_size=CONFIG["batch_size"], shuffle=False, collate_fn=collate_fn)

print(f"[*] Static DataLoaders: {len(loader_static_train)} train batches, {len(loader_static_val)} val batches")
print(f"[*] Dynamic DataLoaders: {len(loader_dynamic_train)} train batches, {len(loader_dynamic_val)} val batches")

In [ ]:
# Cell 7: Training Engine with Validation Checkpointing
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

def evaluate_bert(model, dataloader):
    model.eval()
    all_probs = []
    all_labels = []
    all_conv_ids = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            _, logits = model(input_ids, attention_mask)
            probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()

            all_probs.extend(probs)
            all_labels.extend(labels.cpu().numpy())
            all_conv_ids.extend(batch['conversation_ids'])

    return np.array(all_labels), np.array(all_probs), all_conv_ids

def train_bert_model(model, train_loader, val_loader, epochs, lr, weight_decay, save_path):
    model.to(device)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)

    best_val_macro_f1 = -1.0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0

        for batch_idx, batch in enumerate(train_loader):
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            loss, _ = model(input_ids, attention_mask, labels=labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

            if (batch_idx + 1) % max(1, len(train_loader) // 5) == 0:
                print(f'   Epoch {epoch}/{epochs} | Batch {batch_idx+1}/{len(train_loader)} | Loss: {loss.item():.4f}')

        avg_train_loss = total_loss / len(train_loader)

        # Validation
        val_labels, val_probs, _ = evaluate_bert(model, val_loader)
        val_metrics = evaluate_predictions(val_labels, val_probs, fixed_threshold=0.5)['fixed_threshold_0.5']

        macro_f1 = val_metrics['macro_f1']
        pos_f1 = val_metrics['f1']
        acc = val_metrics['accuracy']

        print(f'[*] Epoch {epoch} Summary | Train Loss: {avg_train_loss:.4f} | Val Acc: {acc:.4f} | Macro-F1: {macro_f1:.4f} | Pos-F1: {pos_f1:.4f}')
        history.append({'epoch': epoch, 'train_loss': avg_train_loss, 'val_metrics': val_metrics})

        if macro_f1 > best_val_macro_f1:
            best_val_macro_f1 = macro_f1
            torch.save(model.state_dict(), save_path)
            print(f'    [+] Checkpoint saved! New best Val Macro-F1: {best_val_macro_f1:.4f}')

    # Load best checkpoint
    model.load_state_dict(torch.load(save_path))
    return model, history

print('[OK] Training engine ready.')

In [ ]:
# Cell 8: Train BERT-SC (Static Baseline)
print('======================================================================')
print('TRAINING BASELINE 1: BERT-SC (Static Full Pre-Target History)')
print('======================================================================')

bert_sc_model = BertForecastingClassifier(pretrained_model_name=CONFIG["model_name"])
static_ckpt_path = CONFIG["output_dir"] / "checkpoints" / "bert_sc_static_best.pt"

bert_sc_model, history_static = train_bert_model(
    model=bert_sc_model,
    train_loader=loader_static_train,
    val_loader=loader_static_val,
    epochs=CONFIG["epochs"],
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    save_path=static_ckpt_path,
)

# Static Evaluation
val_labels_static, val_probs_static, _ = evaluate_bert(bert_sc_model, loader_static_val)
eval_static = evaluate_predictions(val_labels_static, val_probs_static, fixed_threshold=0.5)

print('\n[OK] BERT-SC (Static) Final Validation Performance:')
print(f'    Accuracy:    {eval_static["fixed_threshold_0.5"]["accuracy"]:.4f}')
print(f'    Macro-F1:    {eval_static["fixed_threshold_0.5"]["macro_f1"]:.4f}')
print(f'    Positive-F1: {eval_static["fixed_threshold_0.5"]["f1"]:.4f}')
print(f'    ROC-AUC:     {eval_static["fixed_threshold_0.5"]["roc_auc"]:.4f}')
print(f'    PR-AUC:      {eval_static["fixed_threshold_0.5"]["pr_auc"]:.4f}')

In [ ]:
# Cell 9: Train BERT-SC+ (Dynamic Prefix Baseline)
print('======================================================================')
print('TRAINING BASELINE 2: BERT-SC+ (Dynamic Successive Prefixes)')
print('======================================================================')

bert_sc_plus_model = BertForecastingClassifier(pretrained_model_name=CONFIG["model_name"])
dynamic_ckpt_path = CONFIG["output_dir"] / "checkpoints" / "bert_sc_dynamic_best.pt"

bert_sc_plus_model, history_dynamic = train_bert_model(
    model=bert_sc_plus_model,
    train_loader=loader_dynamic_train,
    val_loader=loader_dynamic_val,
    epochs=CONFIG["epochs"],
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    save_path=dynamic_ckpt_path,
)

# Dynamic Prefix Evaluation
val_labels_dynamic, val_probs_dynamic, _ = evaluate_bert(bert_sc_plus_model, loader_dynamic_val)

df_eval_dynamic = pd.DataFrame(val_dynamic_data)
df_eval_dynamic['predicted_prob'] = val_probs_dynamic

# Forecast Horizon Analysis
fh_metrics = compute_forecast_horizon_metrics(df_eval_dynamic, val_probs_dynamic, threshold=0.5)

# Full-prefix and Max-pooling aggregation
conv_preds = aggregate_dynamic_predictions(df_eval_dynamic, val_probs_dynamic)
eval_full_prefix = evaluate_predictions(conv_preds['label'].to_numpy(), conv_preds['full_prefix_prob'].to_numpy())
eval_max_pooling = evaluate_predictions(conv_preds['label'].to_numpy(), conv_preds['max_prob'].to_numpy())

print('\n[OK] BERT-SC+ (Dynamic) Validation Performance:')
print(f'    [Full-Prefix] Accuracy: {eval_full_prefix["fixed_threshold_0.5"]["accuracy"]:.4f} | Macro-F1: {eval_full_prefix["fixed_threshold_0.5"]["macro_f1"]:.4f}')
print(f'    [Max-Pooling] Accuracy: {eval_max_pooling["fixed_threshold_0.5"]["accuracy"]:.4f} | Macro-F1: {eval_max_pooling["fixed_threshold_0.5"]["macro_f1"]:.4f}')
print(f'    Mean Forecast Horizon: {fh_metrics["mean_horizon"]:.2f} turns')
print(f'    Warned Before Terminal Prefix: {fh_metrics["warned_before_latest_prefix_count"]}/{fh_metrics["positive_conversations_total"]} ({fh_metrics["warned_before_latest_prefix_proportion"]*100:.1f}%)')
print(f'    Negative False Alert Rate:     {fh_metrics["false_alert_conversations_count"]}/{fh_metrics["negative_conversations_total"]} ({fh_metrics["false_alert_rate"]*100:.1f}%)')

In [ ]:
# Cell 10: Compile and Save All Metrics
import json

final_results = {
    "metadata": {
        "model_name": CONFIG["model_name"],
        "epochs": CONFIG["epochs"],
        "batch_size": CONFIG["batch_size"],
        "learning_rate": CONFIG["learning_rate"],
        "max_length": CONFIG["max_length"],
        "seed": CONFIG["seed"],
        "smoke_test": SMOKE_TEST,
        "test_set_accessed": False,
    },
    "bert_sc_static": {
        "training_history": history_static,
        "validation_evaluation": eval_static,
    },
    "bert_sc_plus_dynamic": {
        "training_history": history_dynamic,
        "conversation_full_prefix_evaluation": eval_full_prefix,
        "conversation_max_pooling_evaluation": eval_max_pooling,
        "forecast_horizon_metrics_th_0.5": fh_metrics,
    }
}

metrics_path = CONFIG["output_dir"] / "bert_baselines_validation_metrics.json"
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(final_results, f, indent=2)

print(f'[OK] Validation metrics saved to {metrics_path.resolve()}')

In [ ]:
# Cell 11: Package Outputs for Download
import shutil

zip_filename = 'convoguard_phase2_bert_outputs'
zip_target = shutil.make_archive(zip_filename, 'zip', root_dir=CONFIG["output_dir"])

print('======================================================================')
print('[OK] All Phase 2 artifacts successfully packaged:')
print(f'    ZIP Archive: {zip_target} ({Path(zip_target).stat().st_size:,} bytes)')
print('    Contents:')
print('    - checkpoints/bert_sc_static_best.pt')
print('    - checkpoints/bert_sc_dynamic_best.pt')
print('    - bert_baselines_validation_metrics.json')
print('======================================================================')